In [ ]:
import pandas as pd
import numpy as np
import re
from nltk.translate.bleu_score import sentence_bleu

class ParticipantVisibleError(Exception):
    """
    Custom exception raised when the participant's submission format is incorrect.
    """
    pass

def compute_bleu(reference_text, hypothesis_text):
    """
    Computes the BLEU score between a reference text and a hypothesis text.

    Args:
        reference_text (str): The reference sentence.
        hypothesis_text (str): The generated hypothesis sentence.

    Returns:
        float: The BLEU score (between 0 and 1).
    """
    reference_tokens = reference_text
    hypothesis_tokens = hypothesis_text

    if not hypothesis_tokens:  # Avoid empty hypothesis cases
        return 0.0

    return sentence_bleu([reference_tokens], hypothesis_tokens)

def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str) -> float:
    """
    Computes the average BLEU score for a submission compared to the reference solution.

    Args:
        solution (pd.DataFrame): DataFrame containing the correct reference translations.
        submission (pd.DataFrame): DataFrame containing the submitted translations.
        row_id_column_name (str): The name of the column containing row IDs, which should be ignored.

    Returns:
        float: The average BLEU score for all sentence pairs.

    Raises:
        ParticipantVisibleError: If the submission format is incorrect (wrong number of rows or non-text columns).
    """
    # Remove row ID column
    del solution[row_id_column_name]
    del submission[row_id_column_name]

    # Check column consistency
    if solution.shape != submission.shape:
        raise ParticipantVisibleError("Submission file must have the same number of rows as the solution file.")

    # Ensure all text values are strings
    if not all(solution.dtypes == "object") or not all(submission.dtypes == "object"):
        raise ParticipantVisibleError("All columns must contain text data.")

    # Compute BLEU scores for each sentence pair
    bleu_scores = []
    for ref_text, hyp_text in zip(solution.iloc[:, 0], submission.iloc[:, 0]):
        bleu_score = compute_bleu(ref_text, hyp_text)
        bleu_scores.append(bleu_score)

    # Return average BLEU score
    return np.mean(bleu_scores)
